## Recommandation Model Training

#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity


### Load Datasets

In [2]:
physical_df = pd.read_csv('../../data/preproccedData/Augmented_PreProccedPhysicalActivityParameters.csv')
gym_df = pd.read_csv('../../data/RecommandationDatasets/megaGymDataset.csv')

In [3]:
physical_df.head( )

,Id,Age,Gender,Height,Weight,EnergyLevels,Physical_Activity,Sitting_Time,Cardiovascular_Health,Muscle_Strength,Flexibility,Balance,Thirsty,Pain_or_Discomfort,Available_Time,BMI,DiabetesRisk,PhysicalActivityRisk
0,1.070382,24,1,172.867929,48.426644,2.184795,1.894502,0.938458,0.010730,0.000000,0.000000,0.014709,1.812565,0.004581,48.376007,16.508953,30.774236,85.573579
1,3.223005,24,1,170.783864,53.252082,3.292354,0.913221,0.942123,0.979213,0.000000,0.933494,1.073578,0.918528,0.984561,116.117325,18.634367,32.703477,42.791112
2,4.323651,28,0,158.385838,46.480006,4.376199,1.878412,0.909096,0.000201,0.919126,0.945207,1.086765,1.836717,0.008689,290.311582,18.908881,28.756909,14.256016
3,5.421089,24,0,168.730281,54.230813,2.197266,2.828547,0.938141,0.975245,0.006039,0.920186,1.078358,4.570623,0.000000,116.120954,19.418614,37.049640,42.795648
4,6.487447,22,0,159.408917,44.536954,3.276441,2.818626,0.010396,0.006284,0.904967,0.000000,1.080866,2.732837,0.000000,174.169601,17.876292,24.567752,14.254669


### Data Preprocessing

In [4]:
# Drop rows with missing critical values
gym_df.shape

(2918, 9)

#### Head of the Dataset

In [5]:
gym_df.head()

,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


### Check Missing values

In [6]:
gym_df.isna().sum()

Unnamed: 0       0
Title            0
Desc          1550
Type             0
BodyPart         0
Equipment       32
Level            0
Rating        1887
RatingDesc    2056
dtype: int64

#### Rename Colums

In [7]:
# Clean all column names
gym_df.columns = gym_df.columns.str.strip()                     # Removes leading/trailing spaces and \n
gym_df.columns = gym_df.columns.str.replace('\n', '', regex=True)  # Removes newlines
gym_df.columns = gym_df.columns.str.replace(' ', '_')           # Optional: Replace spaces with underscores

gym_df.rename(columns={
    'Title': 'Exercise_Title',
}, inplace=True)
gym_df.head()

,Unnamed:_0,Exercise_Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


#### Drop irrelevant Colums

In [8]:
gym_df.drop(columns=["Unnamed:_0"], inplace=True)

In [9]:
gym_df.head()

,Exercise_Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


### Check data types


In [10]:
gym_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2918 entries, 0 to 2917
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Exercise_Title  2918 non-null   object 
 1   Desc            1368 non-null   object 
 2   Type            2918 non-null   object 
 3   BodyPart        2918 non-null   object 
 4   Equipment       2886 non-null   object 
 5   Level           2918 non-null   object 
 6   Rating          1031 non-null   float64
 7   RatingDesc      862 non-null    object 
dtypes: float64(1), object(7)
memory usage: 182.5+ KB


#### Handle the missing values

In [11]:
gym_df['Desc'] = gym_df['Desc'].fillna('No description available')
gym_df['Equipment'] = gym_df['Equipment'].fillna('Bodyweight')
gym_df['RatingDesc'] = gym_df['RatingDesc'].fillna('Rating')

# For 'Rating', fill missing with the median rating (or 0 if you prefer)
gym_df['Rating'] = gym_df['Rating'].fillna(gym_df['Rating'].median())

In [12]:
gym_df.head()

,Exercise_Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,Rating
1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,7.9,Rating
2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,7.9,Rating
3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,7.9,Rating
4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,7.9,Rating


In [13]:
gym_df.to_csv("../../data/RecommandationDatasets/updatedgymrecommandations.csv", index=False)

In [13]:
physical_df.head()

,Id,Age,Gender,Height,Weight,EnergyLevels,Physical_Activity,Sitting_Time,Cardiovascular_Health,Muscle_Strength,Flexibility,Balance,Thirsty,Pain_or_Discomfort,Available_Time,BMI,DiabetesRisk,PhysicalActivityRisk
0,1.070382,24,1,172.867929,48.426644,2.184795,1.894502,0.938458,0.010730,0.000000,0.000000,0.014709,1.812565,0.004581,48.376007,16.508953,30.774236,85.573579
1,3.223005,24,1,170.783864,53.252082,3.292354,0.913221,0.942123,0.979213,0.000000,0.933494,1.073578,0.918528,0.984561,116.117325,18.634367,32.703477,42.791112
2,4.323651,28,0,158.385838,46.480006,4.376199,1.878412,0.909096,0.000201,0.919126,0.945207,1.086765,1.836717,0.008689,290.311582,18.908881,28.756909,14.256016
3,5.421089,24,0,168.730281,54.230813,2.197266,2.828547,0.938141,0.975245,0.006039,0.920186,1.078358,4.570623,0.000000,116.120954,19.418614,37.049640,42.795648
4,6.487447,22,0,159.408917,44.536954,3.276441,2.818626,0.010396,0.006284,0.904967,0.000000,1.080866,2.732837,0.000000,174.169601,17.876292,24.567752,14.254669


### Clean and preprocess exercise data

In [14]:
gym_df.dropna(subset=['Exercise_Title', 'Type', 'BodyPart', 'Level'], inplace=True)
gym_df.reset_index(drop=True, inplace=True)

### One-hot encode categorical features

In [15]:
encoder = OneHotEncoder()
exercise_features = encoder.fit_transform(gym_df[['Type', 'BodyPart', 'Level']]).toarray()
exercise_matrix = exercise_features

#### # Map user profile to health goals


In [16]:
def get_user_goals(user):
    goals = []
    if user['Muscle_Strength'] < 0.5:
        goals.append(('Strength', 'Abdominals'))
    if user['Flexibility'] < 0.5:
        goals.append(('Stretching', 'Lower Back'))
    if user['Balance'] < 0.5:
        goals.append(('Balance', 'Legs'))
    if user['EnergyLevels'] < 2.5:
        goals.append(('Cardio', 'Full Body'))
    if not goals:
        goals.append(('Strength', 'Abdominals'))
    return goals

### Score exercises based on user profile

In [17]:
def score_exercises(user):
    goals = get_user_goals(user)
    scores = []
    for _, row in gym_df.iterrows():
        score = 0
        for goal_type, goal_bodypart in goals:
            if goal_type.lower() in row['Type'].lower():
                score += 1
            if goal_bodypart.lower() in row['BodyPart'].lower():
                score += 1
        # Match time availability to difficulty
        if user['Available_Time'] < 60 and row['Level'].lower() == 'beginner':
            score += 1
        if user['Available_Time'] >= 60 and row['Level'].lower() == 'intermediate':
            score += 1
        scores.append(score)
    return np.array(scores)

### Maximal Marginal Relevance (MMR)

In [18]:
def apply_mmr(scores, top_n=5, lambda_param=0.7):
    selected = []
    remaining = list(range(len(scores)))
    similarity_matrix = cosine_similarity(exercise_matrix)

    while len(selected) < top_n and remaining:
        if not selected:
            next_idx = np.argmax(scores[remaining])
            selected.append(remaining.pop(next_idx))
        else:
            mmr_scores = []
            for i in remaining:
                relevance = scores[i]
                diversity = max(similarity_matrix[i][selected]) if selected else 0
                mmr = lambda_param * relevance - (1 - lambda_param) * diversity
                mmr_scores.append(mmr)
            best_idx = remaining[np.argmax(mmr_scores)]
            selected.append(best_idx)
            remaining.remove(best_idx)

    return selected

#### Assign reps/duration for each exercise based on weakness

In [19]:
def assign_workload(user, selected_indices):
    total_time = user['Available_Time']
    plan = []
    
    # Weakness scores: higher = weaker
    muscle_w = 1 - user['Muscle_Strength']
    flex_w = 1 - user['Flexibility']
    balance_w = 1 - user['Balance']
    energy_w = 2.5 - user['EnergyLevels']
    if energy_w < 0: energy_w = 0

    weights = {'Strength': muscle_w, 'Stretching': flex_w, 'Balance': balance_w, 'Cardio': energy_w}
    total_weight = sum(weights.values()) if sum(weights.values()) > 0 else 1

    time_allocs = []
    for idx in selected_indices:
        ex_type = gym_df.iloc[idx]['Type']
        weight = weights.get(ex_type, 0.5)
        portion = weight / total_weight
        time_allocs.append(portion * total_time)

    # Build plan
    for idx, minutes in zip(selected_indices, time_allocs):
        row = gym_df.iloc[idx]
        workload = {
            'Exercise_Title': row['Exercise_Title'],
            'Type': row['Type'],
            'BodyPart': row['BodyPart'],
            'Level': row['Level'],
            'Equipment': row['Equipment'],
            'Recommended_Duration_Minutes': round(minutes, 1),
            'Recommended_Reps': int((minutes * 60) // 30)  # e.g., 1 rep = ~30s
        }
        plan.append(workload)
    
    return pd.DataFrame(plan)

### Recommend exercise plan for a user

In [20]:
def recommend_plan(user_row):
    scores = score_exercises(user_row)
    selected_indices = apply_mmr(scores, top_n=5, lambda_param=0.7)
    plan = assign_workload(user_row, selected_indices)
    return plan

### Example for user 0

In [21]:
user = physical_df.iloc[0]
recommendation_plan = recommend_plan(user)

### Show plan

In [22]:
print(recommendation_plan)


                   Exercise_Title        Type    BodyPart     Level  \
0          Bench barbell roll-out    Strength  Abdominals  Beginner   
1                Dancer's Stretch  Stretching  Lower Back  Beginner   
2                  Stomach Vacuum  Stretching  Abdominals  Beginner   
3  Stiff Leg Barbell Good Morning    Strength  Lower Back  Beginner   
4               Barbell Side Bend    Strength  Abdominals  Beginner   

    Equipment  Recommended_Duration_Minutes  Recommended_Reps  
0     Barbell                          14.7                29  
1  Bodyweight                          14.7                29  
2   Body Only                          14.7                29  
3     Barbell                          14.7                29  
4     Barbell                          14.7                29  
